# NiyamTrace-X — Final Paper Finisher

This notebook consumes:

- `NTX_Q1_14_ICLR_TACL_FINAL_VALIDATION_RESULTS.zip`
- `NiyamTrace-X_All_Three_Formats_Source.zip`

It updates only claims that the validation notebook marked `SUPPORTED`, compiles IEEE/Springer/Elsevier PDFs, audits citations/references, and creates one final paper package.

In [ ]:
from pathlib import Path
from datetime import datetime
import os,sys,json,re,zipfile,shutil,hashlib,subprocess
import pandas as pd
import numpy as np

BASE=Path("/content/NTX_PAPER_FINISHER") if Path("/content").exists() else Path.cwd()/"NTX_PAPER_FINISHER"
WORK=BASE/"work"; OUT=BASE/"final"
for p in [BASE,WORK,OUT]:p.mkdir(parents=True,exist_ok=True)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""):h.update(c)
    return h.hexdigest()

def locate(name):
    for p in [Path.cwd()/name,Path("/content")/name,BASE/name]:
        if p.exists():return p
    return None

In [ ]:
RESULT_ZIP=locate("NTX_Q1_14_ICLR_TACL_FINAL_VALIDATION_RESULTS.zip")
SOURCE_ZIP=locate("NiyamTrace-X_All_Three_Formats_Source.zip")
if RESULT_ZIP is None:raise FileNotFoundError("Upload NTX_Q1_14_ICLR_TACL_FINAL_VALIDATION_RESULTS.zip")
if SOURCE_ZIP is None:raise FileNotFoundError("Upload NiyamTrace-X_All_Three_Formats_Source.zip")

RDIR=WORK/"results";SDIR=WORK/"sources"
shutil.rmtree(RDIR,ignore_errors=True);shutil.rmtree(SDIR,ignore_errors=True)
RDIR.mkdir();SDIR.mkdir()
with zipfile.ZipFile(RESULT_ZIP) as z:z.extractall(RDIR)
with zipfile.ZipFile(SOURCE_ZIP) as z:z.extractall(SDIR)
print("Result SHA",sha256_file(RESULT_ZIP))
print("Source SHA",sha256_file(SOURCE_ZIP))

In [ ]:
claims=pd.read_csv(RDIR/"22_claim_evidence_checklist.csv")
display(claims)
supported=set(claims.loc[claims.status=="SUPPORTED","claim"].astype(str))
external_supported=[x for x in supported if any(k in x for k in ["BFCL","MLCL","AgentDojo","AgentDyn","MCP","τ³","External validation"])]
ep=RDIR/"20_unified_evidence_matrix.csv"
evidence=pd.read_csv(ep) if ep.exists() else pd.DataFrame()
paper_ev=evidence[evidence["paper_eligible"]==True].copy() if len(evidence) and "paper_eligible" in evidence.columns else pd.DataFrame()
print("Supported external claims:",external_supported)

In [ ]:
def tex_escape(s):
    s=str(s)
    for a,b in [("\\","\\textbackslash{}"),("&","\\&"),("%","\\%"),("_","\\_"),("#","\\#")]:
        s=s.replace(a,b)
    return s

ext=paper_ev[paper_ev.evidence_type.astype(str).str.contains("external")] if len(paper_ev) else pd.DataFrame()

if len(ext):
    lines=[
        "\\section{External Cross-Benchmark Validation}",
        "\\label{sec:external}",
        "We additionally evaluate the retained authorization pipeline on external benchmarks for which the official runner produced benchmark-native, paper-eligible scores. We keep unlike benchmark metrics separate and do not average them into a single global accuracy.",
        "",
        "\\begin{table*}[t]",
        "\\centering",
        "\\caption{Paper-eligible external benchmark evidence produced by the final validation notebook. Metrics remain benchmark-native.}",
        "\\label{tab:external}",
        "\\begin{tabular}{llllr}",
        "\\toprule",
        "Benchmark & Model & Metric & Score & $n$\\\\",
        "\\midrule",
    ]
    for _,r in ext.sort_values(["benchmark","model"]).iterrows():
        score="" if pd.isna(r.score) else f"{float(r.score):.4f}"
        n="" if pd.isna(r.n) else str(int(r.n))
        lines.append(f"{tex_escape(r.benchmark)} & {tex_escape(r.model)} & {tex_escape(r.metric)} & {score} & {n} \\\\")
    lines += [
        "\\bottomrule","\\end{tabular}","\\end{table*}","",
        "The external results are interpreted as transfer evidence rather than as a replacement for the frozen internal holdout. Benchmarks or model adapters that did not complete official scoring remain excluded from the quantitative claim set."
    ]
else:
    lines=[
        "\\section{External Validation Status}",
        "\\label{sec:external}",
        "The final validation harness attempted external benchmark execution, but no benchmark-native external score met the paper-eligibility gate in the retained run. We therefore retain the internal frozen and controlled hardening results as the quantitative evidence base and keep external generalization as an explicit limitation."
    ]
external_tex="\n".join(lines)+"\n"
(OUT/"external_validation.tex").write_text(external_tex)
print(external_tex[:2500])

In [ ]:
for fmt in ["ieee","springer","elsevier"]:
    d=SDIR/fmt
    body=(d/"body.tex").read_text()
    abstract=(d/"abstract.tex").read_text()
    marker="\\section{Repository and Runtime Audit}"
    if marker in body and "\\label{sec:external}" not in body:
        body=body.replace(marker,"\\input{external_validation.tex}\n\n"+marker,1)

    if len(ext):
        replacement=(
            "\\paragraph{External benchmark scope.}\n"
            "The final validation harness produced paper-eligible benchmark-native scores for a subset of external benchmarks. "
            "We report those results in Section~\\ref{sec:external} and continue to exclude benchmarks whose official runner, scorer, mapping, or provider adapter did not complete. "
            "Because the external suites use heterogeneous task definitions and metrics, we do not pool them into one global accuracy or safety number."
        )
        if "External validation further" not in abstract:
            names=", ".join(sorted(set(tex_escape(x) for x in ext.benchmark.astype(str))))
            abstract=abstract.rstrip()+"\nExternal validation further yields paper-eligible benchmark-native evidence on "+names+"; unlike benchmark metrics are reported separately rather than collapsed into a single score.\n"
    else:
        replacement=(
            "\\paragraph{External benchmark validation remains incomplete.}\n"
            "The final validation harness did not produce a benchmark-native external score that passed the paper-eligibility gate in the retained run. "
            "Accordingly, all external benchmark attempts remain excluded from quantitative claims, and external generalization remains an open validation target."
        )

    pattern=r"\\paragraph\{External benchmark validation[^}]*\}\n.*?(?=\n\\paragraph\{Execution boundary\})"
    body=re.sub(pattern, lambda _m: replacement+"\n", body, flags=re.S)

    (d/"body.tex").write_text(body)
    (d/"abstract.tex").write_text(abstract)
    shutil.copy2(OUT/"external_validation.tex",d/"external_validation.tex")
print("Patched all three formats.")

In [ ]:
# Preserve the verified bibliography and prebuilt .bbl files from the current source package.
# The generated external-validation section contains no new citation keys; therefore no BibTeX
# regeneration is needed for this finisher.
for fmt in ["ieee","springer","elsevier"]:
    d=SDIR/fmt
    if not (d/"references.bib").exists():
        raise FileNotFoundError(f"Missing references.bib for {fmt}")
    if not (d/"main.bbl").exists():
        print(f"WARNING: {fmt}/main.bbl is missing; BibTeX may be required.")
print("Existing bibliography preserved.")

In [ ]:
if shutil.which("pdflatex") is None:
    if shutil.which("apt-get") is None:
        raise RuntimeError("pdflatex unavailable and apt-get unavailable.")
    p=subprocess.run(
        ["bash","-lc",
         "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
         "texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-pictures texlive-science texlive-publishers"],
        capture_output=True,text=True
    )
    (OUT/"latex_install.log").write_text(p.stdout+"\\nSTDERR\\n"+p.stderr)
    if p.returncode:
        raise RuntimeError("LaTeX installation failed.")
if shutil.which("pdflatex") is None:
    raise RuntimeError("pdflatex still unavailable.")
print("pdflatex:",shutil.which("pdflatex"))

In [ ]:
build_rows=[]
for fmt in ["ieee","springer","elsevier"]:
    d=SDIR/fmt
    # Clean transient cross-reference files, but KEEP main.bbl.
    for pat in ["*.aux","*.blg","*.fdb_latexmk","*.fls","*.log","*.out","*.spl"]:
        for q in d.glob(pat):
            q.unlink(missing_ok=True)

    combined=[]
    returncode=0
    for pass_no in range(1,4):
        p=subprocess.run(["pdflatex","-interaction=nonstopmode","-halt-on-error","main.tex"],
                         cwd=d,capture_output=True,text=True,errors="replace")
        combined.append(f"===== PASS {pass_no} =====\\n"+p.stdout+"\\nSTDERR\\n"+p.stderr)
        returncode=p.returncode
        if returncode!=0:
            break

    (OUT/f"{fmt}_build.log").write_text("\\n".join(combined))
    pdf=d/"main.pdf"
    build_rows.append({"format":fmt,"returncode":returncode,"pdf_exists":pdf.exists(),
                       "size":pdf.stat().st_size if pdf.exists() else 0,
                       "bbl_present":(d/"main.bbl").exists()})
    if returncode==0 and pdf.exists():
        shutil.copy2(pdf,OUT/f"NiyamTrace-X_{fmt.upper()}_FINAL.pdf")

build=pd.DataFrame(build_rows)
build.to_csv(OUT/"build_status.csv",index=False)
display(build)
if not all(r["returncode"]==0 and r["pdf_exists"] for r in build_rows):
    raise RuntimeError("At least one paper format failed to compile.")

In [ ]:
audit=[]
for fmt in ["ieee","springer","elsevier"]:
    d=SDIR/fmt
    # Audit the FINAL TeX log, not the concatenated multi-pass console log.
    final_log=(d/"main.log").read_text(errors="ignore") if (d/"main.log").exists() else ""
    body=(d/"body.tex").read_text()
    absx=(d/"abstract.tex").read_text()
    bib=(d/"references.bib").read_text()

    cites=set()
    for txt in [body,absx]:
        for m in re.finditer(r"\\cite\{([^}]+)\}",txt):
            cites.update(k.strip() for k in m.group(1).split(","))
    keys=set(re.findall(r"@\w+\{([^,]+),",bib))
    unresolved=sorted(cites-keys)

    undef_ref = bool(re.search(r"(Reference .* undefined|There were undefined references)", final_log, flags=re.I))
    undef_cite = bool(re.search(r"(Citation .* undefined|There were undefined citations)", final_log, flags=re.I))

    audit.append({
        "format":fmt,
        "unresolved_citations":len(unresolved),
        "unresolved_keys":";".join(unresolved),
        "undefined_refs":int(undef_ref),
        "undefined_citations":int(undef_cite),
        "overfull_hbox_count":final_log.count("Overfull \\hbox"),
    })

aud=pd.DataFrame(audit)
aud.to_csv(OUT/"paper_audit.csv",index=False)
display(aud)
if aud.unresolved_citations.sum()>0 or aud.undefined_refs.sum()>0 or aud.undefined_citations.sum()>0:
    raise RuntimeError("Citation/reference audit failed.")

In [ ]:
optional=[]
for venue in ["ICLR","TACL"]:
    z=locate(f"{venue}_OFFICIAL_TEMPLATE.zip")
    optional.append({"venue":venue,"template_found":bool(z),"status":"READY_FOR_WRAPPER" if z else "NOT_SUPPLIED"})
pd.DataFrame(optional).to_csv(OUT/"optional_venue_template_status.csv",index=False);display(pd.DataFrame(optional))

In [ ]:
provenance={"created_at":datetime.now().isoformat(),"validation_results_zip_sha256":sha256_file(RESULT_ZIP),
            "source_zip_sha256":sha256_file(SOURCE_ZIP),"supported_claims":sorted(supported),
            "supported_external_claims":sorted(external_supported),"compiled_formats":["IEEE","Springer","Elsevier"]}
(OUT/"FINAL_PROVENANCE.json").write_text(json.dumps(provenance,indent=2))
for fmt in ["ieee","springer","elsevier"]:
    target=OUT/f"{fmt}_source";shutil.rmtree(target,ignore_errors=True);shutil.copytree(SDIR/fmt,target)
for name in ["22_claim_evidence_checklist.csv","20_unified_evidence_matrix.csv","21_hierarchical_summary.csv",
             "21_leave_one_benchmark_out.csv","MANUSCRIPT_FINAL_INTEGRATION.md","FINAL_MANIFEST.json"]:
    p=RDIR/name
    if p.exists():shutil.copy2(p,OUT/name)
zipout=BASE/"NiyamTrace-X_FINAL_PAPER_ALL_FORMATS.zip"
if zipout.exists():zipout.unlink()
with zipfile.ZipFile(zipout,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():z.write(p,arcname=str(p.relative_to(OUT)))
print(zipout,sha256_file(zipout),round(zipout.stat().st_size/1024**2,3),"MiB")
try:
    from google.colab import files;files.download(str(zipout))
except Exception:pass